# Limpieza del dataset ENMT — Encuesta Nacional de Movilidad y Transporte

**Proyecto:** transporte_UNAM

**Responsable de esta etapa:** Juan Pablo Venegas

**Entradas:** `data/enmt_unam.csv` (microdatos) · `data/diccionario_enmt.xls` (libro de códigos SPSS)

**Salidas:** `data/processed/enmt_limpio.csv`, `data/processed/enmt_analitico.csv`, `data/processed/diccionario.json`, `data/processed/reporte_calidad.csv`

---

## Por qué existe este notebook

El archivo original **no tiene un solo `NaN`**, pero eso no significa que no tenga datos faltantes: la encuesta viene de SPSS y codifica los faltantes como **números centinela** (`-1`, `8`, `9`, `98`, `99`, `999-9`). Si el equipo carga el CSV tal cual y calcula un promedio, esos códigos entran como si fueran respuestas reales y contaminan todo el análisis.

El trabajo de este notebook es, entonces:

1. Leer el archivo con la codificación correcta (no es UTF-8).
2. Usar el libro de códigos para saber **qué código significa "faltante" en cada variable** — no adivinando.
3. Convertir esos códigos a `NaN` de verdad.
4. Quitar columnas sin información y **datos personales**.
5. Dejar tipos, nombres y texto en un estado consistente.
6. Exportar un dataset completo y uno reducido, más el diccionario en JSON.

> **Criterio general:** este notebook *no imputa* ni *borra filas* de encuestados. En datos de encuesta, un faltante es información (alguien no supo o no quiso contestar) y las filas están ligadas a un factor de expansión. Lo que hacemos es **hacer visible** lo que estaba oculto, y dejar la decisión de imputar a quien haga el análisis.

## 1. Configuración

Rutas relativas a la raíz del proyecto, para que el notebook corra igual desde `scripts/` o desde la raíz.

In [1]:
import json
import re
import unicodedata
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore", category=FutureWarning)

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 160)
pd.set_option("display.float_format", lambda v: f"{v:,.2f}")

# Raíz del proyecto: funciona si el kernel arranca en scripts/ o en la raíz
BASE = Path.cwd()
if not (BASE / "data").is_dir():
    BASE = BASE.parent
assert (BASE / "data").is_dir(), f"No encuentro data/ desde {Path.cwd()}"

DIR_DATOS = BASE / "data"
DIR_SALIDA = DIR_DATOS / "processed"
DIR_SALIDA.mkdir(parents=True, exist_ok=True)

RUTA_CSV = DIR_DATOS / "enmt_unam.csv"
RUTA_DICC = DIR_DATOS / "diccionario_enmt.xls"

print("Raíz del proyecto :", BASE)
print("CSV crudo         :", RUTA_CSV.name, f"({RUTA_CSV.stat().st_size / 1e6:.2f} MB)")
print("Diccionario       :", RUTA_DICC.name, f"({RUTA_DICC.stat().st_size / 1e6:.2f} MB)")
print("Salidas en        :", DIR_SALIDA)

Raíz del proyecto : /Users/santiagodelac/Tec/Métodos de Razonamiento/articulos/SP-transporte_UNAM
CSV crudo         : enmt_unam.csv (2.34 MB)
Diccionario       : diccionario_enmt.xls (0.41 MB)
Salidas en        : /Users/santiagodelac/Tec/Métodos de Razonamiento/articulos/SP-transporte_UNAM/data/processed


## 2. Carga del dataset crudo

El CSV **no está en UTF-8**. Está en `latin-1` (ISO-8859-1), que es lo que exporta SPSS en español por defecto. Si se intenta leer como UTF-8, `pandas` lanza `UnicodeDecodeError` en el primer acento.

Probamos codificaciones en orden y nos quedamos con la primera que funcione, dejando registro de cuál fue.

In [2]:
def cargar_csv(ruta, codificaciones=("utf-8", "latin-1", "cp1252")):
    """Carga el CSV probando codificaciones en orden. Devuelve (df, codificacion_usada)."""
    for cod in codificaciones:
        try:
            df = pd.read_csv(ruta, encoding=cod, low_memory=False)
            print(f"  ✓ Leído con encoding='{cod}'")
            return df, cod
        except UnicodeDecodeError as e:
            print(f"  ✗ '{cod}' falló: {str(e)[:70]}...")
    raise RuntimeError("Ninguna codificación funcionó")


crudo, ENCODING = cargar_csv(RUTA_CSV)
df = crudo.copy()

FILAS_INI, COLS_INI = df.shape
print(f"\nDimensiones iniciales: {FILAS_INI:,} filas × {COLS_INI:,} columnas")
print(f"Memoria: {df.memory_usage(deep=True).sum() / 1e6:.1f} MB")

  ✗ 'utf-8' falló: 'utf-8' codec can't decode byte 0xe1 in position 6: invalid continuati...
  ✓ Leído con encoding='latin-1'

Dimensiones iniciales: 1,191 filas × 698 columnas
Memoria: 7.5 MB


## 3. Inspección inicial

Antes de tocar nada, un retrato del estado en que llega el archivo.

In [3]:
print("── Tipos de dato ──")
print(df.dtypes.value_counts().to_string())

print("\n── Nulos reales (NaN) ──")
print(f"Celdas nulas: {df.isna().sum().sum():,}  ({df.isna().mean().mean() * 100:.2f}% del total)")
print("→ Cero NaN. Los faltantes están escondidos como códigos numéricos; los sacamos en la sección 6.")

print("\n── Primeras columnas ──")
display(df.iloc[:5, :12])

── Tipos de dato ──
int64      682
object      14
float64      2

── Nulos reales (NaN) ──
Celdas nulas: 0  (0.00% del total)
→ Cero NaN. Los faltantes están escondidos como códigos numéricos; los sacamos en la sección 6.

── Primeras columnas ──


,con1,D_R,edo,muni,loca,folio,ageb,hr_ini1,min_ini1,hr_ter1,min_ter1,dura1
0,1,,28,41,1,5,098-6,9,34,10,14,40
1,2,,26,18,1,1,203-A,13,40,14,0,20
2,3,,15,37,18,4,044-A,12,8,13,10,62
3,4,,15,109,3,5,078-A,11,33,12,24,51
4,5,,12,67,16,4,999-9,10,40,11,0,20


In [4]:
# ¿Qué tan repetidos están los valores en cada columna? Señal temprana de centinelas.
muestra = ["con1", "folio", "edo", "sexo", "edad_1", "escol", "ing_ind", "p2", "p4", "p31"]
resumen = pd.DataFrame({
    "n_unicos": [df[c].nunique() for c in muestra],
    "min": [df[c].min() if pd.api.types.is_numeric_dtype(df[c]) else "-" for c in muestra],
    "max": [df[c].max() if pd.api.types.is_numeric_dtype(df[c]) else "-" for c in muestra],
    "valor_mas_comun": [df[c].value_counts().idxmax() for c in muestra],
    "frec_%": [round(df[c].value_counts(normalize=True).max() * 100, 1) for c in muestra],
}, index=muestra)
print("Valores sospechosos: -1 y 98/99 aparecen como 'min'/'max' en variables donde no tienen sentido.\n")
display(resumen)

Valores sospechosos: -1 y 98/99 aparecen como 'min'/'max' en variables donde no tienen sentido.



,n_unicos,min,max,valor_mas_comun,frec_%
con1,1191,1,1200,1,0.10
folio,25,1,25,5,7.50
edo,25,6,32,15,20.10
sexo,2,1,2,1,53.60
edad_1,6,2,7,3,25.40
escol,6,1,8,3,38.70
ing_ind,6,-1,5,0,49.50
p2,8,0,98,2,61.60
p4,5,1,99,2,46.90
p31,13,0,99,8,23.60


## 4. Libro de códigos: qué significa cada variable y cada número

El `.xls` es un *codebook* exportado de SPSS. Su estructura es irregular (bloques por variable, con celdas vacías como separadores), así que lo recorremos fila por fila con una pequeña máquina de estados.

De cada variable sacamos dos cosas:

- **Etiqueta**: el texto de la pregunta (`p4` → *"¿Qué tanto le gusta caminar?"*).
- **Valores etiquetados**: el catálogo de códigos (`p4` → `1: Mucho, 2: Algo, 3: Poco, 4: Nada`).

Ese catálogo es la pieza clave: nos dice **en qué variables el `8` significa "No sabe"** y en cuáles es una respuesta legítima. Sin él, convertir todos los `8` a `NaN` destruiría datos válidos.

In [5]:
def parsear_diccionario(ruta):
    """Recorre el codebook SPSS y devuelve (etiquetas, valores_etiquetados)."""
    hoja = pd.read_excel(ruta, header=None, names=["a", "b", "c"])
    etiquetas, valores, actual = {}, {}, None
    encabezados = ("conjunto de datos", "libro de códigos", "[conjunto")

    for a, b, c in hoja.itertuples(index=False):
        a_txt = str(a).strip() if pd.notna(a) else ""

        # Fila con solo la columna A → nombre de una variable nueva
        if a_txt and pd.isna(b) and pd.isna(c):
            if a_txt.lower().startswith(encabezados):
                continue
            actual = a_txt
            valores.setdefault(actual, {})
            continue

        if actual is None:
            continue

        # Etiqueta de la variable
        if a_txt == "Atributos estándar" and str(b).strip() == "Etiqueta":
            etiquetas[actual] = str(c).strip() if pd.notna(c) else ""
        # Catálogo de códigos (la primera fila trae "Valores etiquetados", las siguientes vienen vacías)
        elif a_txt in ("Valores etiquetados", "") and pd.notna(b) and pd.notna(c):
            valores[actual][str(b).strip()] = str(c).strip()

    return etiquetas, valores


ETIQUETAS, VALORES = parsear_diccionario(RUTA_DICC)

print(f"Variables con etiqueta          : {len(ETIQUETAS):,}")
print(f"Variables con catálogo de valores: {sum(1 for v in VALORES.values() if v):,}")
print(f"Columnas en el CSV               : {df.shape[1]:,}")

en_csv, en_dicc = set(df.columns), set(ETIQUETAS)
print(f"\nEn el CSV pero no en el diccionario ({len(en_csv - en_dicc)}): {sorted(en_csv - en_dicc)}")
print(f"En el diccionario pero no en el CSV ({len(en_dicc - en_csv)}): {sorted(en_dicc - en_csv)}")

Variables con etiqueta          : 688
Variables con catálogo de valores: 607
Columnas en el CSV               : 698

En el CSV pero no en el diccionario (10): ['h10_15', 'h14_15g', 'h14_8n', 'h17_15', 'h17_8', 'h20_15', 'h22_15', 'h23_1_15', 'h23_1_8', 'h26_2_15']
En el diccionario pero no en el CSV (0): []


In [6]:
# Ejemplos de lo que acabamos de extraer
for v in ["p4", "p1a_1", "sexo", "edad_1", "p31"]:
    print(f"\n{v} — {ETIQUETAS.get(v, '(sin etiqueta)')[:95]}")
    print(f"   {VALORES.get(v, {})}")


p4 — 4 ¿Qué tanto le gusta caminar? (Leer opciones)
   {'1': 'Mucho', '2': 'Algo', '3': 'Poco', '4': 'Nada', '98': 'NS', '99': 'NC'}

p1a_1 — 1a  ¿Con que frecuencia utilizas los siguientes medios de transporte? Tren
   {'1': 'Cotidianamente', '2': 'Ocasionalmente', '3': 'Nunca', '8': 'NS', '9': 'NC'}

sexo — Sexo
   {'1': 'Hombre', '2': 'Mujer'}

edad_1 — Edad
   {'1': 'Menos de 15 años', '2': 'De 15 a 24 años', '3': 'De 25 a 34 años', '4': 'De 35 a 44 años', '5': 'De 45 a 54 años', '6': 'De 55 a 64 años', '7': '65 años y más'}

p31 — 31 En una escala de calificación como en la escuela donde 0 es nada y 10 es mucho, ¿Qué tanto c
   {'98': 'NS', '99': 'NC'}


## 5. Normalización de nombres de columnas

Los nombres vienen mezclados: `D_R`, `Region`, `Tam_loc`, `Pondi2` conviven con `p1a_1` y `h26_13_15`. Los pasamos todos a `snake_case` ASCII para que nadie en el equipo tenga que recordar si `Region` lleva mayúscula.

Guardamos el mapa `original → nuevo` y **re-indexamos el diccionario** con los nombres nuevos, para que siga sirviendo después del renombrado.

In [7]:
def a_snake_case(nombre: str) -> str:
    """'Tam_loc' -> 'tam_loc'; 'Pondi2' -> 'pondi2'; quita acentos y caracteres raros."""
    txt = unicodedata.normalize("NFKD", str(nombre)).encode("ascii", "ignore").decode()
    txt = re.sub(r"(?<=[a-z0-9])(?=[A-Z])", "_", txt)   # camelCase -> camel_Case
    txt = re.sub(r"[^0-9a-zA-Z]+", "_", txt)            # separadores -> _
    txt = re.sub(r"_+", "_", txt).strip("_").lower()
    return txt


MAPA_NOMBRES = {c: a_snake_case(c) for c in df.columns}

# Verificar que no se generen colisiones
colisiones = pd.Series(list(MAPA_NOMBRES.values())).value_counts()
colisiones = colisiones[colisiones > 1]
assert colisiones.empty, f"Nombres duplicados tras normalizar: {colisiones.to_dict()}"

cambiados = {k: v for k, v in MAPA_NOMBRES.items() if k != v}
print(f"Columnas renombradas: {len(cambiados)} de {len(MAPA_NOMBRES)}")
for k, v in cambiados.items():
    print(f"  {k:>12}  →  {v}")

df = df.rename(columns=MAPA_NOMBRES)

# Re-indexar diccionario con los nombres nuevos
ETIQUETAS = {MAPA_NOMBRES.get(k, a_snake_case(k)): v for k, v in ETIQUETAS.items()}
VALORES = {MAPA_NOMBRES.get(k, a_snake_case(k)): v for k, v in VALORES.items()}
print("\n✓ Diccionario re-indexado con los nombres nuevos")

Columnas renombradas: 6 de 698
           D_R  →  d_r
        Region  →  region
       Tam_loc  →  tam_loc
        Pondi2  →  pondi2
       Pondi_v  →  pondi_v
       Pondi_h  →  pondi_h

✓ Diccionario re-indexado con los nombres nuevos


## 6. Nulos ocultos: convertir códigos centinela a `NaN`

Este es el paso central. Definimos tres reglas, de la más segura a la más específica:

| Regla | Qué convierte | Por qué es segura |
|---|---|---|
| **A. `-1` global** | Todos los `-1` en columnas numéricas | Verificamos contra el diccionario que **`-1` nunca aparece como código etiquetado** en ninguna de las 607 variables con catálogo. Es el "no aplica" por salto de pregunta. |
| **B. NS/NC por variable** | Los códigos que el diccionario etiqueta como `NS`, `NC`, `No aplica` | Se aplica **variable por variable**, según su propio catálogo. En `p4` el `8` no se toca (no existe); en `p1a_1` sí (`8: NS`). |
| **B2. `97` no documentado** | El código `97` en variables que ya usan `98: NS` / `99: NC` | El diccionario **etiqueta explícitamente `97: No aplica` en `p15_*`**, pero omite esa fila en otras variables de la misma familia. Es un hueco del codebook, no un dato. |
| **C. Centinelas de texto** | `''`, `'999-9'` (`ageb`) | Códigos de no-respuesta en columnas de texto. |

Lo que **no** hacemos: convertir a ciegas todos los `8`, `9`, `98`, `99`. En variables como `p31`
(escala 0–10) o `p5` (minutos caminando) esos números pueden ser respuestas reales.

In [8]:
# Regla A — comprobar que -1 nunca es un código válido antes de convertirlo
etiquetados_menos_uno = [v for v, cat in VALORES.items() if "-1" in cat]
print(f"Variables donde '-1' es un código etiquetado: {len(etiquetados_menos_uno)}")
assert not etiquetados_menos_uno, "¡'-1' sí es un valor válido en alguna variable! Revisar antes de convertir."
print("✓ Confirmado: '-1' nunca es respuesta válida → es 'no aplica' (salto de pregunta)")

Variables donde '-1' es un código etiquetado: 0
✓ Confirmado: '-1' nunca es respuesta válida → es 'no aplica' (salto de pregunta)


In [9]:
# Regla B — detectar, por variable, qué códigos significan faltante
PATRON_FALTANTE = re.compile(r"^(ns|nc|ne)$|no\s*sabe|no\s*contest|no\s*especif|no\s*aplica", re.I)

CODIGOS_FALTANTE = {}
for var, catalogo in VALORES.items():
    codigos = [c for c, etiqueta in catalogo.items() if PATRON_FALTANTE.match(etiqueta.strip())]
    if codigos:
        CODIGOS_FALTANTE[var] = codigos

etiquetas_detectadas = sorted({
    e.strip() for cat in VALORES.values() for c, e in cat.items()
    if PATRON_FALTANTE.match(e.strip())
})
print(f"Etiquetas reconocidas como faltante: {etiquetas_detectadas}")
print(f"Variables con al menos un código de faltante: {len(CODIGOS_FALTANTE):,}")

ejemplo = dict(list(CODIGOS_FALTANTE.items())[:6])
print(f"\nEjemplos: {ejemplo}")
print(f"\nContraste — p4 ('Mucho/Algo/Poco/Nada') no tiene códigos de faltante: "
      f"{CODIGOS_FALTANTE.get('p4', 'ninguno')} ✓ (su '4' = 'Nada' es respuesta real)")

Etiquetas reconocidas como faltante: ['NC', 'NS', 'Nc', 'No aplica', 'Ns']
Variables con al menos un código de faltante: 445

Ejemplos: {'p1a_1': ['8', '9'], 'p1a_2': ['8', '9'], 'p1a_3': ['8', '9'], 'p1a_4': ['8', '9'], 'p1a_5': ['8', '9'], 'p1a_6': ['8', '9']}

Contraste — p4 ('Mucho/Algo/Poco/Nada') no tiene códigos de faltante: ['98', '99'] ✓ (su '4' = 'Nada' es respuesta real)


### Regla B2 — el `97` que el diccionario olvidó documentar

La primera corrida detectó `97` en las 14 variables `p18_*` (escalas de calificación 0–10), donde por definición ese valor es imposible. No es un error de captura: el codebook **sí documenta `97: No aplica`** en la familia `p15_*`, pero omitió esa fila en `p18_*` y otras.

Es decir, `97` es el "no aplica" estándar del instrumento (a quien no usa el metro no se le pide calificar el metro) y el diccionario está incompleto. Lo tratamos como faltante, pero **solo** en variables que ya usan la convención `98: NS` / `99: NC`, para no tocar variables donde `97` pudiera ser un valor real.

In [10]:
# Evidencia: el diccionario sí documenta 97 en la familia p15_*
print("p15_1 (documentada):", VALORES.get("p15_1", {}))
print("p18_1 (incompleta) :", VALORES.get("p18_1", {}))

CODIGO_NO_APLICA = 97
candidatas_97 = []
for var, catalogo in VALORES.items():
    if var not in df.columns or not pd.api.types.is_numeric_dtype(df[var]):
        continue
    # Solo variables que ya usan la convención 98/99 y NO documentan el 97
    usa_convencion = {"98", "99"}.issubset(set(catalogo))
    if usa_convencion and str(CODIGO_NO_APLICA) not in catalogo:
        if (df[var] == CODIGO_NO_APLICA).any():
            candidatas_97.append(var)

print(f"\nVariables con '97' no documentado: {len(candidatas_97)}")
print(f"  {candidatas_97}")

# Confirmar que 97 es imposible en ellas: ¿está muy lejos del resto de los valores?
if candidatas_97:
    v = candidatas_97[0]
    otros = sorted(df.loc[df[v] != CODIGO_NO_APLICA, v].dropna().unique())
    print(f"\n  Valores de '{v}' sin contar el 97: {otros}")
    print(f"  → El 97 no pertenece a esa escala. Confirmado como 'No aplica'.")

p15_1 (documentada): {'1': 'Siempre', '2': 'Casi siempre', '3': 'Casi nunca', '4': 'Nunca', '97': 'No aplica', '98': 'NS', '99': 'NC'}
p18_1 (incompleta) : {'98': 'NS', '99': 'NC'}

Variables con '97' no documentado: 135
  ['p1c_1_1', 'p1c_1_2', 'p1c_1_3', 'p1c_1_4', 'p1c_1_5', 'p1c_1_6', 'p1c_1_7', 'p1c_1_8', 'p1c_2_1', 'p1c_2_2', 'p1c_2_3', 'p1c_2_4', 'p1c_2_5', 'p1c_2_6', 'p1c_2_7', 'p1c_2_8', 'p1c_3_1', 'p1c_3_2', 'p1c_3_3', 'p1c_3_4', 'p1c_3_5', 'p1c_3_6', 'p1c_3_7', 'p1c_3_8', 'p1c_4_1', 'p1c_4_2', 'p1c_4_3', 'p1c_4_4', 'p1c_4_5', 'p1c_4_6', 'p1c_4_7', 'p1c_4_8', 'p1c_5_1', 'p1c_5_2', 'p1c_5_3', 'p1c_5_4', 'p1c_5_5', 'p1c_5_6', 'p1c_5_7', 'p1c_5_8', 'p1c_6_1', 'p1c_6_2', 'p1c_6_3', 'p1c_6_4', 'p1c_6_5', 'p1c_6_6', 'p1c_6_7', 'p1c_6_8', 'p1c_7_1', 'p1c_7_2', 'p1c_7_3', 'p1c_7_4', 'p1c_7_5', 'p1c_7_6', 'p1c_7_7', 'p1c_7_8', 'p1c_8_1', 'p1c_8_2', 'p1c_8_3', 'p1c_8_4', 'p1c_8_5', 'p1c_8_6', 'p1c_8_7', 'p1c_8_8', 'p1c_9_1', 'p1c_9_2', 'p1c_9_3', 'p1c_9_4', 'p1c_9_5', 'p1c_9_6', 'p1c_9

In [11]:
nulos_antes = df.isna().sum().sum()
cols_numericas = df.select_dtypes(include=[np.number]).columns
cols_texto = df.select_dtypes(include="object").columns

registro = []  # auditoría por columna

# ── Regla A: -1 → NaN
n_menos_uno = int((df[cols_numericas] == -1).sum().sum())
for c in cols_numericas:
    n = int((df[c] == -1).sum())
    if n:
        registro.append({"columna": c, "regla": "A: -1 (no aplica)", "codigos": "-1", "celdas": n})
df[cols_numericas] = df[cols_numericas].replace(-1, np.nan)

# ── Regla B: códigos NS/NC según el catálogo de cada variable
n_ns_nc = 0
for var, codigos in CODIGOS_FALTANTE.items():
    if var not in df.columns:
        continue
    if var in cols_numericas:
        num = [float(c) for c in codigos if re.fullmatch(r"-?\d+", c)]
        if not num:
            continue
        marca = df[var].isin(num)
    else:
        marca = df[var].astype(str).str.strip().isin(codigos)
    n = int(marca.sum())
    if n:
        n_ns_nc += n
        registro.append({"columna": var, "regla": "B: NS/NC (diccionario)",
                         "codigos": ",".join(codigos), "celdas": n})
        df.loc[marca, var] = np.nan

# ── Regla B2: código 97 ("No aplica") no documentado
n_97 = 0
for var in candidatas_97:
    marca = df[var] == CODIGO_NO_APLICA
    n = int(marca.sum())
    if n:
        n_97 += n
        registro.append({"columna": var, "regla": "B2: 97 no aplica (no documentado)",
                         "codigos": "97", "celdas": n})
        df.loc[marca, var] = np.nan

# ── Regla C: centinelas de texto
n_texto = 0
for c in cols_texto:
    if c not in df.columns:
        continue
    marca = df[c].astype(str).str.strip().isin(["", "999-9", "nan"])
    n = int(marca.sum())
    if n:
        n_texto += n
        registro.append({"columna": c, "regla": "C: centinela de texto",
                         "codigos": "'' / 999-9", "celdas": n})
        df.loc[marca, c] = np.nan

auditoria_centinelas = pd.DataFrame(registro)
nulos_despues = df.isna().sum().sum()

print(f"Regla A  — '-1' (no aplica)        : {n_menos_uno:>9,} celdas")
print(f"Regla B  — NS/NC según diccionario : {n_ns_nc:>9,} celdas")
print(f"Regla B2 — '97' no documentado     : {n_97:>9,} celdas")
print(f"Regla C  — centinelas de texto     : {n_texto:>9,} celdas")
print(f"{'─' * 48}")
print(f"NaN antes  : {nulos_antes:>9,}")
print(f"NaN después: {nulos_despues:>9,}   ({nulos_despues / df.size * 100:.1f}% del dataset)")
print(f"\nColumnas afectadas: {auditoria_centinelas['columna'].nunique():,}")

Regla A  — '-1' (no aplica)        :   548,939 celdas
Regla B  — NS/NC según diccionario :     6,064 celdas
Regla B2 — '97' no documentado     :    19,061 celdas
Regla C  — centinelas de texto     :    10,336 celdas
────────────────────────────────────────────────
NaN antes  :         0
NaN después:   584,304   (70.3% del dataset)

Columnas afectadas: 662


In [12]:
# Códigos observados que el diccionario no documenta: los reportamos, NO los convertimos.
sin_documentar = []
for var in df.select_dtypes(include=[np.number]).columns:
    catalogo = VALORES.get(var)
    if not catalogo:
        continue
    validos = {float(c) for c in catalogo if re.fullmatch(r"-?\d+", c)}
    if not validos:
        continue
    observados = set(df[var].dropna().unique())
    # Solo tiene sentido para variables categóricas (pocos valores), no para conteos/escalas
    if len(observados) > len(validos) + 3:
        continue
    huerfanos = observados - validos
    if huerfanos:
        sin_documentar.append({
            "columna": var,
            "codigos_no_documentados": sorted(huerfanos),
            "filas": int(df[var].isin(huerfanos).sum()),
        })

codigos_huerfanos = pd.DataFrame(sin_documentar).sort_values("filas", ascending=False) \
    if sin_documentar else pd.DataFrame(columns=["columna", "codigos_no_documentados", "filas"])

print(f"Variables con códigos no documentados en el diccionario: {len(codigos_huerfanos)}")
if len(codigos_huerfanos):
    display(codigos_huerfanos.head(15))
print("\n⚠ Se dejan tal cual, a propósito. Caso conocido: ing_ind e ing_fam usan 0 sin etiqueta,")
print("  que en contexto significa 'sin ingreso declarado', no un faltante. Convertirlos sería")
print("  una decisión de análisis, no de limpieza.")

Variables con códigos no documentados en el diccionario: 71


,columna,codigos_no_documentados,filas
67,tam_loc,[1],622
68,ing_ind,[0.0],589
66,region,[1],348
53,p8,"[1.0, 2.0, 3.0, 4.0, 8.0]",292
69,ing_fam,[0],264
70,escol,[1.0],146
56,p25_2_2,"[1.0, 2.0, 3.0, 4.0, 5.0]",73
36,p1c_15_1,"[6.0, 8.0, 9.0, 10.0]",27
62,h7,"[1.0, 2.0, 3.0, 4.0]",26
14,p1c_7_7,"[6.0, 7.0, 8.0, 9.0, 10.0]",25



⚠ Se dejan tal cual, a propósito. Caso conocido: ing_ind e ing_fam usan 0 sin etiqueta,
  que en contexto significa 'sin ingreso declarado', no un faltante. Convertirlos sería
  una decisión de análisis, no de limpieza.


## 7. Duplicados

Tres niveles de revisión:

1. **Filas idénticas** en todas las columnas.
2. **Identificador de encuestado** (`con1`): debe ser único, uno por persona.
3. **`folio`**: sospechoso, porque solo tiene 25 valores distintos para ~1,200 filas.

In [13]:
dup_completas = int(df.duplicated().sum())
print(f"1. Filas completamente duplicadas : {dup_completas}")

print(f"\n2. con1 — valores únicos: {df['con1'].nunique():,} / {len(df):,} filas")
print(f"   Duplicados: {int(df['con1'].duplicated().sum())}")
es_id = df["con1"].is_unique
print(f"   → {'✓ con1 es el identificador único de encuestado' if es_id else '✗ con1 NO es único'}")

print(f"\n3. folio — valores únicos: {df['folio'].nunique():,} / {len(df):,} filas")
print("   folio NO identifica al encuestado. Cruzándolo con la geografía:")
por_folio = df.groupby("folio").agg(
    filas=("con1", "size"), estados=("edo", "nunique"), municipios=("muni", "nunique")
).head(8)
display(por_folio)
print("   → folio se repite entre estados distintos: es un consecutivo dentro del")
print("     conglomerado muestral, no un ID global. La llave real de vivienda es")
print("     la combinación edo+muni+loca+ageb+folio.")

1. Filas completamente duplicadas : 0

2. con1 — valores únicos: 1,191 / 1,191 filas
   Duplicados: 0
   → ✓ con1 es el identificador único de encuestado

3. folio — valores únicos: 25 / 1,191 filas
   folio NO identifica al encuestado. Cruzándolo con la geografía:


,filas,estados,municipios
folio,,,
1,89,25,45
2,88,25,45
3,88,25,45
4,89,25,45
5,89,25,45
6,89,25,45
7,89,25,45
8,89,25,45


   → folio se repite entre estados distintos: es un consecutivo dentro del
     conglomerado muestral, no un ID global. La llave real de vivienda es
     la combinación edo+muni+loca+ageb+folio.


In [14]:
# Verificar que la llave compuesta sí identifica viviendas de forma consistente
llave = ["edo", "muni", "loca", "ageb", "folio"]
n_viviendas = df.groupby(llave, dropna=False).ngroups
print(f"Viviendas distintas según {'+'.join(llave)}: {n_viviendas:,}")
print(f"Encuestados por vivienda: {len(df) / n_viviendas:.2f} en promedio")

if dup_completas:
    df = df.drop_duplicates().reset_index(drop=True)
    print(f"\nEliminadas {dup_completas} filas duplicadas")
else:
    print("\n✓ No hay filas que eliminar por duplicidad")

Viviendas distintas según edo+muni+loca+ageb+folio: 1,191
Encuestados por vivienda: 1.00 en promedio

✓ No hay filas que eliminar por duplicidad


## 8. Columnas sin información y datos personales

Tres tipos de columna que salen del dataset:

- **Vacías**: 100 % `NaN` después del paso 6. No aportan nada.
- **Constantes**: un solo valor en todas las filas. Varianza cero, sin poder informativo.
- **Datos personales (PII)**: las columnas `h8_*` contienen **nombres de pila reales** de los integrantes del hogar. No tienen valor analítico y no deben estar en un repositorio compartido. Se eliminan; el conteo de integrantes se conserva en `n_ind`.

In [15]:
# ── PII: nombres de los integrantes del hogar
cols_pii = [c for c in df.columns if re.fullmatch(r"h8_\d+", c)]
print(f"Columnas con nombres de personas (PII): {len(cols_pii)}")
print(f"  {cols_pii}")
print(f"  Etiqueta en el diccionario: {ETIQUETAS.get(cols_pii[0], '(sin etiqueta)')[:80]}")
# Muestra enmascarada: el notebook se versiona, no podemos imprimir los nombres reales
muestra_pii = [n[0] + "*" * (len(n) - 1) for n in df[cols_pii[0]].dropna().head(4)]
print(f"  Ejemplo de contenido (enmascarado): {muestra_pii} … ← son nombres reales")
print(f"  Se conserva 'n_ind' (integrantes del hogar): {df['n_ind'].notna().sum():,} valores")

Columnas con nombres de personas (PII): 9
  ['h8_1', 'h8_2', 'h8_3', 'h8_4', 'h8_5', 'h8_6', 'h8_7', 'h8_8', 'h8_15']
  Etiqueta en el diccionario: 8 Nombre:
  Ejemplo de contenido (enmascarado): ['P****', 'J********************', 'E****', 'M****'] … ← son nombres reales
  Se conserva 'n_ind' (integrantes del hogar): 1,191 valores


In [16]:
cols_vacias = [c for c in df.columns if df[c].isna().all()]
cols_constantes = [c for c in df.columns if c not in cols_vacias and df[c].nunique(dropna=True) <= 1]

print(f"Vacías (100% NaN)      : {len(cols_vacias)}")
print(f"  {cols_vacias[:12]}{' …' if len(cols_vacias) > 12 else ''}")
print(f"\nConstantes (1 valor)   : {len(cols_constantes)}")
print(f"  {cols_constantes[:12]}{' …' if len(cols_constantes) > 12 else ''}")

a_eliminar = sorted(set(cols_vacias) | set(cols_constantes) | set(cols_pii))
cols_antes = df.shape[1]
df = df.drop(columns=a_eliminar)

print(f"\n{'─' * 52}")
print(f"Columnas eliminadas: {len(a_eliminar)}  ({cols_antes} → {df.shape[1]})")
print(f"  · vacías     : {len(cols_vacias)}")
print(f"  · constantes : {len(cols_constantes)}")
print(f"  · PII        : {len(cols_pii)}")

REGISTRO_ELIMINADAS = pd.DataFrame({
    "columna": a_eliminar,
    "motivo": ["PII (nombre de persona)" if c in cols_pii
               else "vacía (100% NaN)" if c in cols_vacias
               else "constante (1 solo valor)" for c in a_eliminar],
    "etiqueta": [ETIQUETAS.get(c, "") for c in a_eliminar],
})
display(REGISTRO_ELIMINADAS.head(10))

Vacías (100% NaN)      : 67
  ['d_r', 'p1c_20_1', 'p1c_20_2', 'p1c_20_3', 'p25_2_3', 'p25_3_3', 'p25_4_3', 'p25_5_3', 'p25_6_3', 'p25_7_3', 'p25_8_3', 'p25_9_3'] …

Constantes (1 valor)   : 69
  ['resul4', 'p1b_20', 'p25_1_3', 'p25_2_5', 'p25_3_5', 'p25_5_5', 'p25_6_5', 'p25_7_5', 'p25_9_5', 'p25_5_6', 'p25_7_6', 'p25_1_7'] …

────────────────────────────────────────────────────
Columnas eliminadas: 144  (698 → 554)
  · vacías     : 67
  · constantes : 69
  · PII        : 9


,columna,motivo,etiqueta
0,d_r,vacía (100% NaN),<ninguno>
1,h10_15,vacía (100% NaN),
2,h11_15,vacía (100% NaN),11 Edad
3,h12_15,vacía (100% NaN),12 Alfabetismo:
4,h13_15,vacía (100% NaN),13 ¿Asiste a la escuela?
5,h14_15g,constante (1 solo valor),
6,h14_15n,vacía (100% NaN),14 Nivel de instruccion: Grado
7,h15_15,vacía (100% NaN),15 Estado Civil:
8,h15_8,constante (1 solo valor),15 Estado Civil:
9,h16_15,vacía (100% NaN),16 Durante la semana pasada ¿trabajó?


## 9. Corrección de tipos

Después de introducir `NaN`, `pandas` degradó muchas columnas de `int64` a `float64` (porque el `NaN` clásico solo existe en punto flotante). Eso hace que un código de respuesta `3` se imprima como `3.0`, lo cual es feo y confunde.

Usamos el tipo **`Int64` nullable** de pandas: enteros que sí admiten faltantes.

In [17]:
print("Antes:", df.dtypes.value_counts().to_dict())

convertidas = 0
for c in df.select_dtypes(include=["float64"]).columns:
    serie = df[c].dropna()
    if serie.empty:
        continue
    # ¿Todos los valores presentes son enteros?
    if np.allclose(serie, serie.round()):
        df[c] = df[c].round().astype("Int64")
        convertidas += 1

print(f"\nfloat64 → Int64 nullable: {convertidas} columnas")
print("Después:", df.dtypes.value_counts().to_dict())

# Los factores de expansión deben quedarse como decimales
pesos = [c for c in ["pondi2", "pondi_v", "pondi_h"] if c in df.columns]
print(f"\nFactores de expansión (se mantienen decimales): "
      f"{ {c: str(df[c].dtype) for c in pesos} }")

Antes: {dtype('float64'): 518, dtype('int64'): 33, dtype('O'): 3}

float64 → Int64 nullable: 516 columnas
Después: {Int64Dtype(): 516, dtype('int64'): 33, dtype('O'): 3, dtype('float64'): 2}

Factores de expansión (se mantienen decimales): {'pondi2': 'int64', 'pondi_v': 'int64', 'pondi_h': 'int64'}


## 10. Valores inconsistentes en texto libre

`p12` es una pregunta abierta (*"¿bajo qué circunstancias dejaría de usar el automóvil?"*) y viene sucia: mayúsculas mezcladas, acentos inconsistentes, espacios sobrantes y variantes de la misma respuesta (`"descompuesto"`, `"descompostura"`, `"que se descomponga"`).

Normalizamos la forma (minúsculas, sin acentos, espacios colapsados) y agrupamos las variantes más frecuentes en categorías. **La respuesta original se conserva** en una columna aparte por si el equipo quiere revisar la codificación.

In [18]:
def normalizar_texto(s):
    """Minúsculas, sin acentos, sin puntuación sobrante, espacios colapsados."""
    if pd.isna(s):
        return pd.NA
    t = unicodedata.normalize("NFKD", str(s)).encode("ascii", "ignore").decode()
    t = re.sub(r"[^\w\s]", " ", t.lower())
    t = re.sub(r"\s+", " ", t).strip()
    return t or pd.NA


cols_texto_libre = [c for c in df.columns if df[c].dtype == "object"]
print(f"Columnas de texto restantes: {cols_texto_libre}")

if "p12" in df.columns:
    df["p12_original"] = df["p12"]
    df["p12"] = df["p12"].apply(normalizar_texto)

    print(f"\np12 — respuestas con texto: {df['p12'].notna().sum():,}")
    print(f"      variantes distintas   : {df['p12'].nunique():,}")
    print("\nMás frecuentes tras normalizar:")
    display(df["p12"].value_counts().head(10).to_frame("frecuencia"))

Columnas de texto restantes: ['ageb', 'p12', 'p12a_1']

p12 — respuestas con texto: 1,142
      variantes distintas   : 173

Más frecuentes tras normalizar:


,frecuencia
p12,
1,899
que se descomponga,13
cuando no circula,8
ninguna,7
descompuesto,6
no sirve,5
descompostura,5
cuando se descompone,4
por el trafico,4


In [19]:
# Agrupar variantes de la misma idea en categorías temáticas
REGLAS_P12 = {
    "descompostura_del_auto": r"descompost|descompon|descompues|se descompon|fall(a|e)",
    "mejor_transporte_publico": r"transporte publico|mejor(a|e)|eficien|seguro el transporte",
    "costo_gasolina":           r"gasolina|combustible|caro|costo|gasto|economi",
    "trafico_congestion":       r"trafico|congestion|embotell",
    "distancia_cercania":       r"distancia|cerca|cercan",
    "salud_incapacidad":        r"salud|incapacidad|enferm|edad|vista",
    "restriccion_vehicular":    r"no circula|hoy no|verificacion",
    "ninguna_circunstancia":    r"^ningun|nada|por nada|siempre lo",
}


def clasificar_p12(txt):
    if pd.isna(txt):
        return pd.NA
    for categoria, patron in REGLAS_P12.items():
        if re.search(patron, txt):
            return categoria
    return "otra"


if "p12" in df.columns:
    df["p12_categoria"] = df["p12"].apply(clasificar_p12)
    resumen_p12 = df["p12_categoria"].value_counts(dropna=False).to_frame("frecuencia")
    resumen_p12["%"] = (resumen_p12["frecuencia"] / df["p12"].notna().sum() * 100).round(1)
    display(resumen_p12)

    ETIQUETAS["p12_categoria"] = "p12 recodificada en categorías temáticas (derivada)"
    ETIQUETAS["p12_original"] = "p12 respuesta textual sin normalizar (derivada)"
    print("→ Se agregaron 'p12_original' y 'p12_categoria'. La normalización redujo la dispersión")
    print("  del texto sin perder el dato crudo.")

,frecuencia,%
p12_categoria,,
otra,1007,88.20
descompostura_del_auto,62,5.40
<NA>,49,4.30
costo_gasolina,20,1.80
restriccion_vehicular,14,1.20
ninguna_circunstancia,13,1.10
trafico_congestion,10,0.90
distancia_cercania,8,0.70
mejor_transporte_publico,6,0.50


→ Se agregaron 'p12_original' y 'p12_categoria'. La normalización redujo la dispersión
  del texto sin perder el dato crudo.


## 11. Valores atípicos y fuera de rango

Distinguimos dos cosas que suelen confundirse:

- **Fuera de rango (imposible)**: una hora de `27`, una escala 0–10 con valor `15`. Es un error de captura. Se convierte a `NaN`.
- **Atípico estadístico**: alguien que declara 12 viajes en un día. Es raro pero **posible**, y en una encuesta puede ser justo el caso interesante. Solo se **reporta**, no se toca.

Borrar atípicos en datos de encuesta sesga los resultados, sobre todo con factores de expansión.

In [20]:
# Rangos válidos por definición de la variable o del instrumento
RANGOS = {
    "p2":  (0, 30,  "viajes realizados ayer"),
    "p5":  (0, 240, "minutos dispuesto a caminar"),
    "p8":  (0, 15,  "automóviles en casa"),
    "p31": (0, 10,  "escala de cumplimiento 0-10"),
    "sd2": (12, 110, "edad del informante"),
    "h3":  (0, 20,  "cuartos para dormir"),
    "h4":  (1, 30,  "cuartos en la vivienda"),
    "h5":  (1, 60,  "personas en la vivienda"),
    "h7":  (1, 20,  "hogares en la vivienda"),
}
# Las escalas 0-10 de calificación del transporte (p18_*)
for c in df.columns:
    if re.fullmatch(r"p18_\d+", c):
        RANGOS[c] = (0, 10, "escala de calificación 0-10")
# Horas y minutos de la entrevista
for c in df.columns:
    if re.fullmatch(r"hr_(ini|ter)\d", c):
        RANGOS[c] = (0, 23, "hora de la entrevista")
    elif re.fullmatch(r"min_(ini|ter)\d", c):
        RANGOS[c] = (0, 59, "minuto de la entrevista")

filas_rango = []
total_fuera = 0
for col, (lo, hi, desc) in RANGOS.items():
    if col not in df.columns:
        continue
    fuera = df[col].notna() & ((df[col] < lo) | (df[col] > hi))
    n = int(fuera.sum())
    if n:
        total_fuera += n
        filas_rango.append({"columna": col, "descripcion": desc, "rango": f"[{lo}, {hi}]",
                            "fuera_de_rango": n,
                            "valores": sorted(df.loc[fuera, col].dropna().unique())[:6]})
        df.loc[fuera, col] = np.nan

fuera_de_rango = pd.DataFrame(filas_rango)
print(f"Variables con reglas de rango: {len([c for c in RANGOS if c in df.columns])}")
print(f"Celdas fuera de rango → NaN  : {total_fuera:,}")
if len(fuera_de_rango):
    display(fuera_de_rango)
else:
    print("✓ Ninguna variable viola su rango definido: la captura fue consistente.")

Variables con reglas de rango: 39
Celdas fuera de rango → NaN  : 0
✓ Ninguna variable viola su rango definido: la captura fue consistente.


In [21]:
# Atípicos estadísticos: solo se reportan
def reportar_atipicos(serie, k=1.5):
    s = pd.to_numeric(serie, errors="coerce").dropna()
    if s.nunique() < 5:
        return None
    q1, q3 = s.quantile([0.25, 0.75])
    iqr = q3 - q1
    if iqr == 0:
        return None
    lo, hi = q1 - k * iqr, q3 + k * iqr
    n = int(((s < lo) | (s > hi)).sum())
    return {"columna": serie.name, "n_atipicos": n, "%": round(n / len(s) * 100, 1),
            "limite_inf": round(lo, 1), "limite_sup": round(hi, 1),
            "min": s.min(), "max": s.max()}


candidatas = [c for c in ["p2", "p5", "p8", "p9", "p31", "sd2", "h3", "h4", "h5", "n_ind"]
              if c in df.columns]
reporte = [r for c in candidatas if (r := reportar_atipicos(df[c]))]
atipicos = pd.DataFrame(reporte).sort_values("%", ascending=False)
display(atipicos)
print("→ Se reportan pero NO se eliminan: en encuestas son respuestas válidas y las filas")
print("  están ligadas a un factor de expansión. Quitarlas sesgaría las estimaciones.")

,columna,n_atipicos,%,limite_inf,limite_sup,min,max
2,p9,26,11.40,"-4,750.00","13,250.00",100,50000
3,p31,31,3.30,4.00,12.00,0,10
5,h3,39,3.30,0.50,4.50,1,20
6,h4,22,1.80,0.00,8.00,1,20
7,h5,14,1.20,-1.00,7.00,1,50
1,p5,8,0.70,-20.00,60.00,0,120
4,sd2,6,0.50,-5.00,83.00,15,97
8,n_ind,6,0.50,-1.00,7.00,1,14
0,p2,3,0.30,-3.00,5.00,0,6


→ Se reportan pero NO se eliminan: en encuestas son respuestas válidas y las filas
  están ligadas a un factor de expansión. Quitarlas sesgaría las estimaciones.


## 12. Dataset analítico reducido

El dataset completo tiene cientos de columnas del *roster* del hogar (`h9_*` … `h26_*`, repetidas hasta 15 veces) y de matrices de preguntas que quedaron casi vacías después de limpiar los centinelas. Para el análisis del equipo armamos una versión reducida con dos criterios:

1. **Bloques temáticos relevantes**: identificación, geografía, factores de expansión, sociodemográficos, y las preguntas de movilidad a nivel individuo.
2. **Cobertura mínima**: descartar lo que tenga más de **70 % de faltantes**, porque con esa cobertura no se sostiene ningún cruce.

In [22]:
BLOQUES = {
    "identificacion": [r"^con1$"],
    "geografia":      [r"^(edo|muni|loca|ageb|folio|estrato|region|tam_loc)$"],
    "ponderadores":   [r"^pondi"],
    "sociodemografico": [r"^(sexo|edad_1|escol|cond_act|ing_ind|ing_fam|est_civil|"
                         r"period_pago|n_ind|sd\d+a?)$"],
    "vivienda":       [r"^h[1-7]$"],
    "uso_de_modos":   [r"^p1a_\d+$"],
    "caminata":       [r"^(p3_\d+|p3a_\d+|p4|p5)$"],
    "automovil":      [r"^(p6|p7|p8|p9|p10|p11|p12|p12_categoria|p13|p14)$"],
    "seguridad_vial": [r"^(p15_\d+|p26|p27|p28|p29|p31)$"],
    "transporte_publico": [r"^(p16|p17_\d+|p18_\d+|p19|p20|p21|p21a|p22|p23|p24)$"],
    "discapacidad":   [r"^(p32|p33|p34)$"],
}

seleccion, origen = [], {}
for bloque, patrones in BLOQUES.items():
    for c in df.columns:
        if c not in origen and any(re.fullmatch(p, c) for p in patrones):
            seleccion.append(c)
            origen[c] = bloque

print(f"Columnas que caen en algún bloque temático: {len(seleccion)}")
print(pd.Series(origen).value_counts().to_string())

Columnas que caen en algún bloque temático: 140
caminata              32
transporte_publico    27
uso_de_modos          22
sociodemografico      16
seguridad_vial        14
automovil             10
geografia              8
vivienda               7
discapacidad           3
identificacion         1


In [31]:
UMBRAL_NULOS = 0.70
cobertura = df[seleccion].isna().mean()
descartadas = cobertura[cobertura > UMBRAL_NULOS].sort_values(ascending=False)
finales = [c for c in seleccion if c not in descartadas.index]

print(f"Descartadas por >{UMBRAL_NULOS:.0%} de faltantes: {len(descartadas)}")
if len(descartadas):
    tabla = descartadas.head(10).to_frame("% nulos")
    tabla["% nulos"] = (tabla["% nulos"] * 100).round(1)
    tabla["etiqueta"] = [ETIQUETAS.get(c, "")[:65] for c in tabla.index]
    display(tabla)

analitico = df[finales].copy()
print(f"\nDataset analítico: {analitico.shape[0]:,} filas × {analitico.shape[1]:,} columnas")
print(f"Faltantes promedio: {analitico.isna().mean().mean() * 100:.1f}%")

Descartadas por >70% de faltantes: 32


,% nulos,etiqueta
p3a_15,99.40,3a ¿Cuánto tiempo camina normalmente para …? ...
h7,97.80,7 ¿Cuantos hogares o grupos de personas tienen...
p34,96.90,34 ¿Debido a su discapacidad cuál fue el princ...
p33,96.70,33 ¿Qué tipo de discapacidad?
p3a_3,92.50,3a ¿Cuánto tiempo camina normalmente para …? ...
p28,88.20,28 ¿Quién fue el causante?
p3a_14,88.20,3a ¿Cuánto tiempo camina normalmente para …? ...
p29,87.90,29 ¿En qué tipo de transporte viajaba cuando o...
p27,87.60,27 ¿Qué tipo de accidente fue?
p3a_9,83.40,3a ¿Cuánto tiempo camina normalmente para …? ...



Dataset analítico: 1,191 filas × 108 columnas
Faltantes promedio: 7.9%


## 13. Exportación

Cuatro archivos en `data/processed/`:

| Archivo | Qué es |
|---|---|
| `enmt_limpio.csv` | Dataset completo limpio, UTF-8 |
| `enmt_analitico.csv` | Subconjunto para análisis |
| `diccionario.json` | Etiquetas de variable y de valores, con los nombres ya normalizados |
| `reporte_calidad.csv` | Estado columna por columna (tipo, nulos, únicos, etiqueta) |

Los datos se guardan con **códigos numéricos**, no con etiquetas de texto. Es la decisión correcta: los códigos ocupan menos, no se rompen con acentos y se pueden mapear a etiquetas en el momento de graficar usando `diccionario.json`. La última celda muestra cómo hacerlo.

> `data/processed/` está en `.gitignore`: estos archivos se **reproducen** corriendo el notebook. Lo que se versiona es el código y el CSV crudo.

In [24]:
RUTA_LIMPIO = DIR_SALIDA / "enmt_limpio.csv"
RUTA_ANALITICO = DIR_SALIDA / "enmt_analitico.csv"
RUTA_DICC_JSON = DIR_SALIDA / "diccionario.json"
RUTA_REPORTE = DIR_SALIDA / "reporte_calidad.csv"

df.to_csv(RUTA_LIMPIO, index=False, encoding="utf-8")
analitico.to_csv(RUTA_ANALITICO, index=False, encoding="utf-8")

diccionario_salida = {
    "_meta": {
        "fuente": "Encuesta Nacional de Movilidad y Transporte (ENMT)",
        "encoding_original": ENCODING,
        "filas": int(df.shape[0]),
        "columnas_dataset_completo": int(df.shape[1]),
        "columnas_dataset_analitico": int(analitico.shape[1]),
    },
    "variables": {
        c: {
            "etiqueta": ETIQUETAS.get(c, ""),
            "valores": VALORES.get(c, {}),
            "tipo": str(df[c].dtype),
            "pct_nulos": round(float(df[c].isna().mean()) * 100, 2),
            "en_analitico": c in analitico.columns,
        }
        for c in df.columns
    },
}
RUTA_DICC_JSON.write_text(
    json.dumps(diccionario_salida, ensure_ascii=False, indent=2), encoding="utf-8"
)

reporte_calidad = pd.DataFrame({
    "columna": df.columns,
    "tipo": [str(df[c].dtype) for c in df.columns],
    "n_nulos": [int(df[c].isna().sum()) for c in df.columns],
    "pct_nulos": [round(float(df[c].isna().mean()) * 100, 2) for c in df.columns],
    "n_unicos": [int(df[c].nunique(dropna=True)) for c in df.columns],
    "en_analitico": [c in analitico.columns for c in df.columns],
    "etiqueta": [ETIQUETAS.get(c, "") for c in df.columns],
})
reporte_calidad.to_csv(RUTA_REPORTE, index=False, encoding="utf-8")

for r in (RUTA_LIMPIO, RUTA_ANALITICO, RUTA_DICC_JSON, RUTA_REPORTE):
    print(f"  ✓ {r.relative_to(BASE)}  ({r.stat().st_size / 1024:.0f} KB)")

  ✓ data/processed/enmt_limpio.csv  (961 KB)
  ✓ data/processed/enmt_analitico.csv  (280 KB)
  ✓ data/processed/diccionario.json  (198 KB)
  ✓ data/processed/reporte_calidad.csv  (61 KB)


In [25]:
# Verificación: releer lo exportado en UTF-8 y comprobar que coincide
verif = pd.read_csv(RUTA_LIMPIO, encoding="utf-8", low_memory=False)
assert verif.shape == df.shape, f"Dimensiones no coinciden: {verif.shape} vs {df.shape}"
assert list(verif.columns) == list(df.columns), "Los nombres de columna no coinciden"
print(f"✓ enmt_limpio.csv se relee correctamente en UTF-8: {verif.shape[0]:,} × {verif.shape[1]:,}")

verif_a = pd.read_csv(RUTA_ANALITICO, encoding="utf-8", low_memory=False)
assert verif_a.shape == analitico.shape
print(f"✓ enmt_analitico.csv se relee correctamente en UTF-8: {verif_a.shape[0]:,} × {verif_a.shape[1]:,}")

dicc_verif = json.loads(RUTA_DICC_JSON.read_text(encoding="utf-8"))
print(f"✓ diccionario.json válido: {len(dicc_verif['variables']):,} variables")

✓ enmt_limpio.csv se relee correctamente en UTF-8: 1,191 × 556
✓ enmt_analitico.csv se relee correctamente en UTF-8: 1,191 × 108
✓ diccionario.json válido: 556 variables


## 14. Resumen de la limpieza

In [26]:
resumen = pd.DataFrame([
    ("Filas", f"{FILAS_INI:,}", f"{df.shape[0]:,}"),
    ("Columnas (completo)", f"{COLS_INI:,}", f"{df.shape[1]:,}"),
    ("Columnas (analítico)", "—", f"{analitico.shape[1]:,}"),
    ("Codificación", ENCODING, "utf-8"),
    ("Celdas NaN", f"{nulos_antes:,}", f"{df.isna().sum().sum():,}"),
    ("% de faltantes", "0.0%", f"{df.isna().mean().mean() * 100:.1f}%"),
    ("Columnas de texto libre", "14", f"{len(df.select_dtypes('object').columns)}"),
], columns=["Métrica", "Antes", "Después"])
display(resumen.set_index("Métrica"))

convertidos = n_menos_uno + n_ns_nc + n_97 + n_texto + total_fuera
nan_final = int(df.isna().sum().sum())

print(f"""
Faltantes recuperados de códigos centinela
  Regla A  · '-1' no aplica             {n_menos_uno:>9,}
  Regla B  · NS/NC del diccionario      {n_ns_nc:>9,}
  Regla B2 · '97' no documentado        {n_97:>9,}
  Regla C  · centinelas de texto        {n_texto:>9,}
  Regla D  · fuera de rango imposible   {total_fuera:>9,}
  {'-' * 41}
  Celdas convertidas a NaN              {convertidos:>9,}
  menos las que estaban en las
  {len(a_eliminar)} columnas eliminadas             {convertidos - nan_final:>9,}
  {'-' * 41}
  NaN en el dataset final               {nan_final:>9,}

Columnas eliminadas ({len(a_eliminar)})
  vacías (100% NaN)                     {len(cols_vacias):>9}
  constantes (1 solo valor)             {len(cols_constantes):>9}
  PII (nombres de personas)             {len(cols_pii):>9}

Duplicados
  filas idénticas                       {dup_completas:>9}
  con1 (ID de encuestado) único         {'sí' if es_id else 'no':>9}

Filas conservadas: {df.shape[0]:,} de {FILAS_INI:,} (no se eliminó ningún encuestado)
""")

,Antes,Después
Métrica,,
Filas,"1,191","1,191"
Columnas (completo),698,556
Columnas (analítico),—,108
Codificación,latin-1,utf-8
Celdas NaN,0,"421,949"
% de faltantes,0.0%,63.7%
Columnas de texto libre,14,5



Faltantes recuperados de códigos centinela
  Regla A  · '-1' no aplica               548,939
  Regla B  · NS/NC del diccionario          6,064
  Regla B2 · '97' no documentado           19,061
  Regla C  · centinelas de texto           10,336
  Regla D  · fuera de rango imposible           0
  -----------------------------------------
  Celdas convertidas a NaN                584,400
  menos las que estaban en las
  144 columnas eliminadas               162,451
  -----------------------------------------
  NaN en el dataset final                 421,949

Columnas eliminadas (144)
  vacías (100% NaN)                            67
  constantes (1 solo valor)                    69
  PII (nombres de personas)                     9

Duplicados
  filas idénticas                               0
  con1 (ID de encuestado) único                sí

Filas conservadas: 1,191 de 1,191 (no se eliminó ningún encuestado)



In [27]:
# Cómo usar el diccionario para pasar de códigos a etiquetas al momento de analizar
def etiquetar(serie, variable=None):
    """Devuelve la serie con los códigos reemplazados por sus etiquetas de texto."""
    variable = variable or serie.name
    catalogo = VALORES.get(variable, {})
    if not catalogo:
        return serie
    mapa = {float(k): v for k, v in catalogo.items() if re.fullmatch(r"-?\d+", k)}
    return serie.map(lambda x: mapa.get(float(x), x) if pd.notna(x) else x)


ejemplo = pd.DataFrame({
    "codigo": analitico["p4"].head(8).values,
    "etiqueta": etiquetar(analitico["p4"]).head(8).values,
})
print(f"p4 — {ETIQUETAS['p4']}\n")
display(ejemplo)

print("\nDistribución con etiquetas:")
display(etiquetar(analitico["p4"]).value_counts(dropna=False).to_frame("frecuencia"))

p4 — 4 ¿Qué tanto le gusta caminar? (Leer opciones)



,codigo,etiqueta
0,<NA>,NaN
1,1,Mucho
2,4,Nada
3,2,Algo
4,2,Algo
5,2,Algo
6,4,Nada
7,2,Algo



Distribución con etiquetas:


,frecuencia
p4,
Algo,558
Mucho,271
Poco,261
Nada,93
NaN,8


---

### Notas para el equipo

- El dataset limpio guarda **códigos numéricos**. Para graficar, usa la función `etiquetar()` de arriba o carga `data/processed/diccionario.json`.
- **No se imputó nada.** Si tu análisis necesita imputación, hazla en tu propio notebook y documenta el método: así no arrastramos supuestos ajenos.
- **No se borraron filas.** El dataset conserva los 1,191 encuestados. Recuerda usar los factores de expansión (`pondi2`, `pondi_v`, `pondi_h`) para cualquier estimación poblacional.
- Los faltantes altos en varias columnas **no son un error**: vienen de saltos de pregunta del cuestionario (a quien no tiene coche no se le pregunta cuánto gasta en gasolina).
- Si algo del criterio de limpieza no les cuadra, está todo en las reglas A/B/C/D de las secciones 6 y 11 — se pueden ajustar y volver a correr.